[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/composite_strip/composite_strip.ipynb)

# A strip that twists when pulled

Reorder the same four fibre layers and a tensile load either twists the strip or leaves it flat. The strip's strain is a map on directions with its stretch and twist left open, and each ply reads it along and across its own fibres. Pairing each reading with its stress and summing through the thickness builds an energy form; the free edges come in as its Schur complement, and one solve against the pull gives the stretch and the twist.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from numga.extensor import Extensor
from examples.mechanics.composite_strip import render

In [ ]:
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
# Both state components are dimensionless: axial strain and thickness times twist per length.
State = ga.gatype.from_blades("1 yz")
Measurement = ga.gatype((Scalar, State))                      # Scalar <- State
Energy = ga.gatype((Scalar, State, State))                    # Scalar <- (State, State)
Strain = ga.gatype((Vector, Vector, State))                   # Vector <- (Vector, State)
extension: Measurement = mv.scalar([1]).scalar_product(State)  # []
# The minus sign compensates for the negative square of the yz blade.
twist: Measurement = (-mv.yz).scalar_product(State)            # []

## 1. The strain, read along each ply

The strain at a height through the thickness is a map on directions with the state left open: the stretch along the strip, and the shear the twist brings at that height, opposite above and below the middle. Each ply reads it along its fibres, across them, and as their shear, `fibres | strain(fibres)`; turning the ply is the rotor that turns its fibres, with no rotated components to write out.

In [ ]:
# Two stacks of the same four plies, listed from bottom to top.
angles = np.deg2rad([[-45, -45, 45, 45], [-45, 45, 45, -45]])
longitudinal_modulus, transverse_modulus = 130000.0, 10000.0    # MPa, illustrative carbon/epoxy
poisson, shear_modulus = 0.3, 5000.0
length, width, thickness = 100.0, 20.0, 1.0                     # mm
tension = 30.0                                                  # N/mm of width
moduli = (longitudinal_modulus, transverse_modulus, poisson, shear_modulus)

# Two samples per ply integrate its quadratic strain energy exactly.
plies = angles.shape[-1]
ply_thickness = thickness / plies
nodes = np.array([-1, 1]) / np.sqrt(3)
centres = (np.arange(plies) + 0.5) * ply_thickness - thickness / 2
heights: Vector = mv.z * (centres[:, None] + nodes * ply_thickness / 2)  # [plies, samples]
weights = np.full((plies, len(nodes)), ply_thickness / 2)       # [plies, samples], thickness weights


def strain(height: Scalar, thickness: float) -> Strain:
    """The midsurface strain at a height through the thickness, as a map on directions with the
    stretch–twist state left open: the stretch along the strip, and the shear the twist brings at
    that height, opposite above and below the middle."""
    stretch = mv.x * (mv.x | Vector) * extension                                        # [] Vector <- (Vector, State)
    # Twist tilts the normal both ways in the plane; dividing by thickness gives angle per length.
    shear = -height / thickness * (mv.x * (mv.y | Vector) + mv.y * (mv.x | Vector)) * twist  # [...] Vector <- (Vector, State)
    return stretch + shear                                                              # [...] Vector <- (Vector, State)


# Each ply's fibres, turned from the strip's length; the trailing axis meets the depth samples.
fibres: Vector = (mv.xy * (-angles[..., None] / 2)).exp() >> mv.x  # [cases, plies, 1]
transverse: Vector = mv.xy | fibres                              # [cases, plies, 1]
local: Strain = strain(heights | mv.z, thickness)               # [plies, samples]
readings = stack([fibres | local(fibres), transverse | local(transverse),
                  2 * (fibres | local(transverse))])            # [3, cases, plies, samples] Measurement

## 2. Stresses and the energy form

A ply's stresses follow from its readings, the normal ones coupled by Poisson contraction and shear on its own. Each reading times its stress leaves two state slots open, and summed over the three readings and through the thickness that is the energy form: half of `energy(state, state)` is the stored energy per area.

In [ ]:
def ply_stress(along: Extensor, across: Extensor, sliding: Extensor, longitudinal_modulus: float,
               transverse_modulus: float, poisson: float, shear_modulus: float) -> tuple[Extensor, Extensor, Extensor]:
    """A ply's stresses along and across its fibres and in shear, from its strain read the same
    ways, of any arity: the normal stresses coupled by Poisson contraction, shear on its own."""
    denominator = 1 - poisson**2 * transverse_modulus / longitudinal_modulus
    along_stress = (longitudinal_modulus * along + poisson * transverse_modulus * across) / denominator
    across_stress = transverse_modulus * (poisson * along + across) / denominator
    return along_stress, across_stress, shear_modulus * sliding


stresses = stack(ply_stress(*readings, *moduli))                # [3, cases, plies, samples] Measurement
energy: Energy = (readings * stresses * weights).sum(axis=(0, -2, -1))   # [cases]

## 3. Free edges

The width is free to contract. A unit width strain, read the same ways, couples to the state through the stresses and has a stiffness of its own; letting the width relax until its force vanishes takes that coupling out of the energy, its Schur complement.

In [ ]:
widening = mv.y * (mv.y | Vector)                               # [] Vector <- Vector
width_readings = stack([fibres | widening(fibres), transverse | widening(transverse),
                        2 * (fibres | widening(transverse))])   # [3, cases, plies, 1] Scalar
width_stresses = stack(ply_stress(*width_readings, *moduli))    # [3, cases, plies, 1] Scalar
# With the width held fixed, coupling(state) is the transverse force per unit length.
coupling: Measurement = (width_readings * stresses * weights).sum(axis=(0, -2, -1))      # [cases]
width_stiffness: Scalar = (width_readings * width_stresses * weights).sum(axis=(0, -2, -1))  # [cases]
relaxed: Energy = energy - coupling * coupling / width_stiffness  # [cases]

## 4. Pull

Only the axial force is applied; the twist settles through the coupling in the energy. Reordering the plies keeps the stretch but cancels the twist.

In [ ]:
state: State = relaxed.solve(tension * extension)               # [cases]
width_strain: Scalar = -coupling(state) / width_stiffness        # [cases]
twist_angle: Scalar = twist(state) / thickness * length           # [cases], radians over the length

In [ ]:
with np.printoptions(precision=4, suppress=True):
    print("axial strain:     ", extension(state).to_array())
    print("width strain:     ", width_strain.to_array())
    print("twist over length:", np.degrees(twist_angle.to_array()), "degrees")

## 5. The strips under a growing pull

The response is linear, so each frame scales the full one as the pull grows and falls back; the displacements are drawn five times larger. Lines along the top ply's fibres ride on each surface. The symmetric stack stays flat, the antisymmetric one twists.

In [ ]:
def deform(points: Vector, state: State, width_strain: Scalar, thickness: float) -> Vector:
    """The midsurface's uniform stretch, width contraction and small-displacement twist."""
    along: Scalar = points | mv.x                            # [length_samples, width_samples]
    across: Scalar = points | mv.y                           # [length_samples, width_samples]
    # Each state acts on the whole grid; the two trailing axes sample the midsurface.
    state, width_strain = state[..., None, None], width_strain[..., None, None]  # [..., 1, 1] State, Scalar
    # Twist displaces opposite edges in opposite height directions, increasing along the strip.
    return (points + mv.x * along * extension(state) + mv.y * across * width_strain
            + mv.z * along * across * twist(state) / thickness)  # [..., length_samples, width_samples] Vector


magnification = 5.0
fibre_spacing = 7.0                                      # mm between drawn fibre lines
fibre_samples = 9
separation = 1.6 * width                                 # mm between the strips' centres
frame_count, duration_ms = 60, 60
length_samples, width_samples = 61, 13
loads = (1 - np.cos(np.linspace(0, 2 * np.pi, frame_count, endpoint=False))) / 2   # [frames]
reference: Vector = (mv.x * np.linspace(0, length, length_samples)[:, None]
                     + mv.y * np.linspace(-width / 2, width / 2, width_samples)[None, :])  # [length samples, width samples]
# Each strip deforms about its own centreline, then sits beside the other.
placement: Vector = mv.y * (separation * np.array([-0.5, 0.5]))[:, None, None]   # [cases, 1, 1]
frames: Vector = deform(reference, state * (magnification * loads[:, None]),
                        width_strain * (magnification * loads[:, None]), thickness) + placement  # [frames, cases, length samples, width samples]
# Lines along each layup's top ply's fibres, carried by the same deformation.
top: Vector = (mv.xy * (-angles[:, -1] / 2)).exp() >> mv.x                          # [cases]
across = np.linspace(-width / 2, width / 2, fibre_samples)                         # [samples] mm
starts = np.arange(width / 2, length - width / 2, fibre_spacing)                   # [lines] mm along the centre line
fibre_lines: Vector = (mv.x * starts[:, None]
                       + top[:, None, None] * (across / (top | mv.y)[:, None, None]))  # [cases, lines, samples]
fibre_frames: Vector = deform(fibre_lines, state * (magnification * loads[:, None]),
                              width_strain * (magnification * loads[:, None]), thickness) + placement  # [frames, cases, lines, samples]
render.inline(render.animate_strips(reference + placement, frames, fibre_frames), duration_ms)

The antisymmetric stack twists about 7° over its length; the symmetric stack cancels the stretch–twist coupling. Displacements are drawn five times larger. This linear, uniform-curvature model describes the strip interior, away from grips; the chosen balanced layups let the other deformation modes decouple.

In classical laminate matrix notation, this is the axial-load response of the $ABD$ system, with transverse membrane strain eliminated before solving. Symmetry sets the $B$ block to zero; it need not remove bend–twist coupling in $D$. [Stanford and Jutte, *Aeroelastic Tailoring via Tow Steered Composites*, introduction](https://ntrs.nasa.gov/api/citations/20140012777/downloads/20140012777.pdf).